# Notebook 1 – Data exploration and train/validation/test splits

In this notebook we will:

- Load simulation data for one operational regime (Normal / Idling / Shut / Start).
- Explore different strategies for splitting the data into **training**, **validation**, and **test** sets.
- Inspect how these splits affect the **input space** (Wind Speed, TI) and the **target distributions** (e.g. Power, DELs).
- Do some basic **statistical analysis**:
  - Pearson correlations,
  - Descriptive statistics,
  - A simple linear regression baseline.
- Visualize everything interactively with Plotly.

The goal is *not* to tune the “perfect split”, but to **understand how different splitting strategies and target choices influence what the model will see**.


## Parameters and settings

In this cell we define all the parameters that control the analysis:

- `SELECTED_REGIME`: which operating mode (Normal / Idling / Shut / Start) to analyze.
- `TARGET_COL`: which output channel we are currently focusing on (e.g. `PowAc_mean`, `TBFA_DEL4`, `BRFW_DEL10`).
- `SELECTED_INPUT_COLS`: which input features we want to use in the statistical analysis and linear model (Wind Speed, TI, and optionally rotor speed, torque, etc.).
- `SPLIT_METHOD`: which strategy to use for train/validation splitting:
  - `"random"` – random rows to train/val.
  - `"holdout_wsp"` – hold out a wind speed band for validation.
  - `"holdout_ti"` – hold out a TI band for validation.
  - `"stratified_target"` – stratify on binned target values.
  - `"stratified_joint"` – stratify on joint bins of (Wind Speed, TI, target).
- Parameters controlling:
  - train/val fraction,
  - random seed (reproducibility),
  - holdout ranges,
  - number of bins for stratified splits.

If you change any of these values here, you must use the **same values** in Notebook 2 to reproduce the same split.


In [ ]:
# CELL 1  Imports and global configuration

import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = "Data"

# Choose one operational mode at a time for this notebook
SELECTED_REGIME = "Normal"   # options: "Normal", "Idling", "Shut", "Start"

# Column names for inputs (for splitting / stats)
# Students can add/remove here for correlation analysis later
SELECTED_INPUT_COLS = [
    "Wind Speed",
    "Turbulence Intensity",
    "RotSpd_mean",  # you can uncomment this for extra analysis
    # "PitAnBl1_mean",  # you can uncomment this for extra analysis
    # "TrqGen_mean",  # you can uncomment this for extra analysis
]

# Default target column to use for splitting & stats
TARGET_COL = "BRFW_DEL10"    # e.g. "PowAc_mean", "TBFA_DEL4", "TBSS_DEL4", "BRFW_DEL10","BREW_DEL10", "MSBMY_loc_DEL4","MSBMZ_loc_DEL4", "TTTOR_DEL4"
# Main split method to use for the factorial dataset
SPLIT_METHOD = "stratified_target"  # options:
                         # "random", "holdout_wsp", "holdout_ti", "stratified_target", "stratified_joint"

# Fractions and bands
TRAIN_FRACTION = 0.7      # for random / stratified splits
RANDOM_STATE   = 42     # for reproducibility

# Bands for holdout splits
HOLDOUT_WSP_RANGE = (9.0, 11.0)    # [min, max] m/s
HOLDOUT_TI_RANGE  = (0.12, 0.16)   # [min, max]

# Binning setup for stratified splits
TARGET_STRAT_NBINS   = 8   # for stratified_target
WSP_STRAT_NBINS      = 4   # for stratified_joint
TI_STRAT_NBINS       = 3   # for stratified_joint
TARGET_JOINT_NBINS   = 5   # for stratified_joint

PLOT_WIDTH = 900   # adjust 
PLOT_HEIGHT = 700  # adjust 

## Load data for the selected regime

Here we:

- Build the filenames for the selected regime (`<regime>_training_data.csv` and `<regime>_test_data.csv`).
- Load the **full-factorial** training pool (`df_full`) and the **randomized** test set (`df_test`).
- Print shapes and column names as a sanity check.

The training file contains all simulations that we will later split into training and validation sets.  
The test file is kept separate and will be used only for final evaluation of the surrogates.


In [ ]:
# CELL 2: Load data

# ============================
# Load one regime (Normal / Idling / Shut / Start)
# ============================

train_file = os.path.join(DATA_DIR, f"{SELECTED_REGIME}_training_data.csv")
test_file  = os.path.join(DATA_DIR, f"{SELECTED_REGIME}_test_data.csv")

if not os.path.exists(train_file) or not os.path.exists(test_file):
    raise FileNotFoundError(f"Missing files for {SELECTED_REGIME}: "
                            f"{train_file} or {test_file}")

df_full = pd.read_csv(train_file)  # full factorial (train+val pool)
df_test = pd.read_csv(test_file)   # randomized test set

print(f"Loaded regime: {SELECTED_REGIME}")
print("  Full-factorial (train+val pool):", df_full.shape)
print("  Test (randomized):               ", df_test.shape)

# Quick sanity check: columns
print("\nColumns in training file:")
print(df_full.columns.tolist())


Loaded regime: Normal
  Full-factorial (train+val pool): (329, 92)
  Test (randomized):                (94, 92)

Columns in training file:
['Wind Speed', 'Turbulence Intensity', 'RotSpd_min', 'RotSpd_max', 'RotSpd_mean', 'RotSpd_std', 'PitAnBl1_min', 'PitAnBl1_max', 'PitAnBl1_mean', 'PitAnBl1_std', 'TrqGen_min', 'TrqGen_max', 'TrqGen_mean', 'TrqGen_std', 'PowAc_min', 'PowAc_max', 'PowAc_mean', 'PowAc_std', 'TwrClr_min', 'TwrClr_max', 'TwrClr_mean', 'TwrClr_std', 'TBFA_min', 'TBFA_max', 'TBFA_mean', 'TBFA_std', 'TBFA_DEL4', 'TBSS_min', 'TBSS_max', 'TBSS_mean', 'TBSS_std', 'TBSS_DEL4', 'TBTOR_min', 'TBTOR_max', 'TBTOR_mean', 'TBTOR_std', 'TBTOR_DEL4', 'TTFA_min', 'TTFA_max', 'TTFA_mean', 'TTFA_std', 'TTFA_DEL4', 'TTSS_min', 'TTSS_max', 'TTSS_mean', 'TTSS_std', 'TTSS_DEL4', 'TTTOR_min', 'TTTOR_max', 'TTTOR_mean', 'TTTOR_std', 'TTTOR_DEL4', 'BRFW_min', 'BRFW_max', 'BRFW_mean', 'BRFW_std', 'BRFW_DEL10', 'BREW_min', 'BREW_max', 'BREW_mean', 'BREW_std', 'BREW_DEL10', 'BRTOR_min', 'BRTOR_max',

| Short name             | Full name                                         | Unit   |
|------------------------|---------------------------------------------------|--------|
| Wind Speed             | Hub-height wind speed                             | m/s    |
| Turbulence Intensity   | Turbulence intensity at hub height                | %      |
| RotSpd                 | Rotor speed                                       | rpm    |
| PitAnBl1               | Blade 1 pitch angle                               | deg     |
| TrqGen                 | Generator torque                                  | Nm     |
| PowAc                  | Electrical / active power output                  | W      |
| TwrClr                 | Minimum tower clearance (blade–tower distance)    | m      |
| TBFA                   | Tower base fore–aft bending moment                | kNm    |
| TBSS                   | Tower base side–side bending moment               | kNm    |
| TBTOR                  | Tower base torsional moment                       | kNm    |
| TTFA                   | Tower top fore–aft bending moment                 | kNm    |
| TTSS                   | Tower top side–side bending moment                | kNm    |
| TTTOR                  | Tower top torsional moment                        | kNm    |
| BRFW                   | Blade root flap-wise bending moment               | kNm    |
| BREW                   | Blade root edge-wise bending moment               | kNm    |
| BRTOR                  | Blade root torsional moment                       | kNm    |
| MSBMX                  | Main shaft bending moment about x-axis            | kNm    |
| MSBMY                  | Main shaft bending moment about y-axis            | kNm    |
| MSBMZ                  | Main shaft torsion                                | kNm    |
        
**Notes**

- Most columns in the CSV use these base names with suffixes like `_mean`, `_std`, `_min`, `_max`, `_DEL4`, `_DEL10`, e.g. `PowAc_mean`, `TBFA_DEL4`, `MSBMY_glob_std`. These are statistics derived from the simulation timeseries
- `_DEL4` / `_DEL10` are **damage equivalent loads** with Wohler exponent 4 or 10, but the physical unit is still the same as the underlying moment (kNm).  
- `_glob_` vs `_loc_` only changes the coordinate system (global tower-fixed vs. local component frame), not the units.  


## Helper functions for train/validation splits

In this section we define generic functions to split the full-factorial dataset (`df_full`) into **training** and **validation** sets.

We implement several strategies:

- `split_random`: purely random train/val split.
- `split_holdout_range`: hold out a band in one variable (e.g. a wind speed range) as validation.
- `split_stratified_target`: stratify based on **binned target values**, so that train and val see similar target distributions.
- `split_stratified_joint`: stratify based on **joint bins of (Wind Speed, TI, target)**, approximating similar input + target coverage in train and val.

All of these functions take the **full** dataset and return two dataframes: `df_train` and `df_val`.
We will choose which one to use via `SPLIT_METHOD` in the parameter cell.


In [ ]:
# CELL 3: Split helper functions  

# Split helper functions
# ============================

def split_random(df, train_fraction, random_state):
    """
    Simple random split of df into train/val.

    This ignores inputs/target and just shuffles rows.
    """
    df_train, df_val = train_test_split(
        df,
        train_size=train_fraction,
        random_state=random_state,
        shuffle=True,
    )
    return df_train.copy(), df_val.copy()


def split_holdout_range(df, col_name, vmin, vmax):
    """
    Split df into train/val based on a holdout range in one column.

    All rows with col_name in [vmin, vmax] go to val. Others go to train.
    """
    mask_val = (df[col_name] >= vmin) & (df[col_name] <= vmax)
    df_val   = df.loc[mask_val].copy()
    df_train = df.loc[~mask_val].copy()

    print(f"Holdout on {col_name} in [{vmin}, {vmax}]")
    print("  Train size:", df_train.shape[0])
    print("  Val size  :", df_val.shape[0])

    return df_train, df_val


def make_quantile_bins(series, nbins):
    """
    Make quantile-based bins for a 1D series, returning bin labels.

    Each bin should contain roughly the same number of samples.
    """
    # Clip nbins if there are too few unique values
    nbins = min(nbins, series.nunique())
    if nbins <= 1:
        # all samples in one bin
        return np.zeros(len(series), dtype=int)

    # Compute quantile edges (0, 1/nbins, ..., 1)
    quantiles = np.linspace(0, 1, nbins + 1)
    edges = series.quantile(quantiles).values

    # Ensure edges are strictly increasing (avoid duplicates)
    edges = np.unique(edges)
    if len(edges) - 1 < nbins:
        # We had to collapse some bins
        nbins = len(edges) - 1

    # pd.cut will assign bin indices [0, nbins-1]
    bin_labels = pd.cut(series, bins=edges, labels=False, include_lowest=True)

    return bin_labels.astype(int)


def split_stratified_target(df, target_col, train_fraction, nbins, random_state):
    """
    Stratified split on a binned version of the target variable.

    This tries to preserve the target distribution in train and val.
    """
    y = df[target_col]
    # Build quantile-based bins on target
    bins = make_quantile_bins(y, nbins)

    df_train, df_val = train_test_split(
        df,
        train_size=train_fraction,
        random_state=random_state,
        shuffle=True,
        stratify=bins,
    )
    return df_train.copy(), df_val.copy()


def build_joint_strata_labels(df, wsp_col, ti_col, target_col,
                              n_wsp_bins, n_ti_bins, n_target_bins):
    """
    Build joint strata labels by binning wsp, ti, and target.

    Each row gets a label like 'w2_t1_y3' based on its bin indices.
    """
    wsp_bins    = make_quantile_bins(df[wsp_col],    n_wsp_bins)
    ti_bins     = make_quantile_bins(df[ti_col],     n_ti_bins)
    target_bins = make_quantile_bins(df[target_col], n_target_bins)

    labels = [
        f"w{iw}_t{it}_y{iy}"
        for iw, it, iy in zip(wsp_bins, ti_bins, target_bins)
    ]
    return np.array(labels)


def split_stratified_joint(df, wsp_col, ti_col, target_col,
                           train_fraction, n_wsp_bins, n_ti_bins,
                           n_target_bins, random_state):
    """
    Stratified split based on joint bins of (wsp, ti, target).

    This approximates preserving both input and target distributions.
    """
    labels = build_joint_strata_labels(
        df,
        wsp_col=wsp_col,
        ti_col=ti_col,
        target_col=target_col,
        n_wsp_bins=n_wsp_bins,
        n_ti_bins=n_ti_bins,
        n_target_bins=n_target_bins,
    )

    df_train, df_val = train_test_split(
        df,
        train_size=train_fraction,
        random_state=random_state,
        shuffle=True,
        stratify=labels,
    )
    return df_train.copy(), df_val.copy()


In [ ]:
# CELL 4: Plotting functons


def plot_input_scatter_plotly(df_all,
                              wsp_col="Wind Speed",
                              ti_col="Turbulence Intensity",
                              set_col="set",
                              title=None):
    """
    Interactive 2D scatter of inputs (Wind Speed vs Turbulence Intensity),
    colored by dataset (train/val/test).
    """
    if title is None:
        title = "Input space: Wind Speed vs Turbulence Intensity"

    fig = go.Figure()

    for set_name in ["train", "val", "test"]:
        mask = df_all[set_col] == set_name
        if not mask.any():
            continue

        fig.add_trace(go.Scattergl(
            x=df_all.loc[mask, wsp_col],
            y=df_all.loc[mask, ti_col],
            mode="markers",
            name=set_name,
            marker=dict(size=6, opacity=0.6),
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Wind Speed [m/s]",
        yaxis_title="Turbulence Intensity [-]",
        template="plotly_white",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
    )

    fig.show()


def plot_wsp_histogram_plotly(df_train, df_val, df_test,
                              wsp_col="Wind Speed",
                              nbins=40):
    """
    Overlaid histogram for Wind Speed for train/val/test.
    """
    wsp_train = df_train[wsp_col].dropna().values
    wsp_val   = df_val[wsp_col].dropna().values
    wsp_test  = df_test[wsp_col].dropna().values

    vmin = min(wsp_train.min(), wsp_val.min(), wsp_test.min())
    vmax = max(wsp_train.max(), wsp_val.max(), wsp_test.max())

    fig = go.Figure()

    for vals, name in [(wsp_train, "train"),
                       (wsp_val,   "val"),
                       (wsp_test,  "test")]:
        fig.add_trace(go.Histogram(
            x=vals,
            name=name,
            nbinsx=nbins,
            histnorm="probability density",  
            opacity=0.5,
        ))

    fig.update_layout(
        title="Distribution of Wind Speed",
        xaxis_title="Wind Speed [m/s]",
        yaxis_title="Density",
        template="plotly_white",
        barmode="overlay",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
    )

    fig.show()


def plot_ti_histogram_plotly(df_train, df_val, df_test,
                             ti_col="Turbulence Intensity",
                             nbins=40):
    """
    Overlaid histogram for Turbulence Intensity for train/val/test.

    Values are converted to percentage for plotting.
    """
    ti_train = df_train[ti_col].dropna().values * 100.0
    ti_val   = df_val[ti_col].dropna().values * 100.0
    ti_test  = df_test[ti_col].dropna().values * 100.0

    vmin = min(ti_train.min(), ti_val.min(), ti_test.min())
    vmax = max(ti_train.max(), ti_val.max(), ti_test.max())

    fig = go.Figure()

    for vals, name in [(ti_train, "train"),
                       (ti_val,   "val"),
                       (ti_test,  "test")]:
        fig.add_trace(go.Histogram(
            x=vals,
            name=name,
            nbinsx=nbins,
            histnorm="probability density",
            opacity=0.5,
        ))

    fig.update_layout(
        title="Distribution of Turbulence Intensity",
        xaxis_title="Turbulence Intensity [%]",
        yaxis_title="Density",
        template="plotly_white",
        barmode="overlay",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
    )

    fig.show()


def plot_target_histogram_plotly(df_train, df_val, df_test,
                                 target_col, nbins=60):
    """
    Overlaid histogram of the target variable for train/val/test.
    """
    train_vals = df_train[target_col].dropna().values
    val_vals   = df_val[target_col].dropna().values
    test_vals  = df_test[target_col].dropna().values

    vmin = min(train_vals.min(), val_vals.min(), test_vals.min())
    vmax = max(train_vals.max(), val_vals.max(), test_vals.max())

    fig = go.Figure()

    for vals, name in [(train_vals, "train"),
                       (val_vals,   "val"),
                       (test_vals,  "test")]:
        fig.add_trace(go.Histogram(
            x=vals,
            name=name,
            nbinsx=nbins,
            histnorm="probability density",
            opacity=0.5,
        ))

    fig.update_layout(
        title=f"Distribution of {target_col}",
        xaxis_title=target_col,
        yaxis_title="Density",
        template="plotly_white",
        barmode="overlay",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
    )

    fig.show()

def plot_target_vs_inputs_plotly(df,
                                 target_col,
                                 wsp_col="Wind Speed",
                                 ti_col="Turbulence Intensity",
                                 set_col="set"):
    """
    Two interactive scatter plots:

    1) target vs Wind Speed, colored by Turbulence Intensity.
       Separate traces for train/val/test with different markers.

    2) target vs Turbulence Intensity, colored by Wind Speed.
       Separate traces for train/val/test with different markers.
    """
    # Global ranges for color mapping
    ti_all  = df[ti_col].values
    wsp_all = df[wsp_col].values

    ti_min, ti_max   = float(np.nanmin(ti_all)), float(np.nanmax(ti_all))
    wsp_min, wsp_max = float(np.nanmin(wsp_all)), float(np.nanmax(wsp_all))

    marker_symbols = {
        "train": "circle",
        "val":   "x",
        "test":  "triangle-up",
    }

    # ---------------------------
    # Plot 1: target vs Wind Speed (color = TI)
    # ---------------------------
    fig1 = go.Figure()

    for set_name in ["train", "val", "test"]:
        df_set = df[df[set_col] == set_name]
        if df_set.empty:
            continue

        show_scale = (set_name == "train")  # single colorbar

        fig1.add_trace(go.Scattergl(
            x=df_set[wsp_col],
            y=df_set[target_col],
            mode="markers",
            name=set_name,
            marker=dict(
                symbol=marker_symbols.get(set_name, "circle"),
                size=9,                     # bigger markers
                opacity=0.6,
                color=df_set[ti_col],
                cmin=ti_min,
                cmax=ti_max,
                colorscale="Viridis",
                showscale=show_scale,
                colorbar=dict(title="TI [-]") if show_scale else None,
            ),
        ))

    fig1.update_layout(
        title=f"{target_col} vs Wind Speed (color = TI)",
        xaxis_title="Wind Speed [m/s]",
        yaxis_title=target_col,
        template="plotly_white",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor="left",
            yanchor="top",
        ),
    )

    fig1.show()

    # ---------------------------
    # Plot 2: target vs TI (color = WS)
    # ---------------------------
    fig2 = go.Figure()

    for set_name in ["train", "val", "test"]:
        df_set = df[df[set_col] == set_name]
        if df_set.empty:
            continue

        show_scale = (set_name == "train")  # single colorbar

        fig2.add_trace(go.Scattergl(
            x=df_set[ti_col],
            y=df_set[target_col],
            mode="markers",
            name=set_name,
            marker=dict(
                symbol=marker_symbols.get(set_name, "circle"),
                size=9,                     # bigger markers
                opacity=0.6,
                color=df_set[wsp_col],
                cmin=wsp_min,
                cmax=wsp_max,
                colorscale="Viridis",
                showscale=show_scale,
                colorbar=dict(title="Wind Speed [m/s]") if show_scale else None,
            ),
        ))

    fig2.update_layout(
        title=f"{target_col} vs Turbulence Intensity (color = Wind Speed)",
        xaxis_title="Turbulence Intensity [-]",
        yaxis_title=target_col,
        template="plotly_white",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor="left",
            yanchor="top",
        ),
    )

    fig2.show()


def plot_binned_target_vs_wsp_plotly(df_all,
                                     target_col,
                                     wsp_col="Wind Speed",
                                     set_col="set",
                                     nbins=15):
    """
    Compute and plot binned average of target vs Wind Speed for each set
    (train/val/test) as interactive line plots.
    """
    wsp_vals = df_all[wsp_col].values
    wsp_min, wsp_max = wsp_vals.min(), wsp_vals.max()
    bins = np.linspace(wsp_min, wsp_max, nbins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    fig = go.Figure()

    for set_name in ["train", "val", "test"]:
        df_set = df_all[df_all[set_col] == set_name]
        if df_set.empty:
            continue

        indices = np.digitize(df_set[wsp_col].values, bins=bins) - 1
        bin_means = np.full(nbins, np.nan)

        for i in range(nbins):
            mask = indices == i
            if np.any(mask):
                bin_means[i] = df_set.loc[mask, target_col].mean()

        fig.add_trace(go.Scatter(
            x=bin_centers,
            y=bin_means,
            mode="lines+markers",
            name=set_name,
        ))

    fig.update_layout(
        title=f"Binned average of {target_col} vs Wind Speed",
        xaxis_title="Wind Speed [m/s]",
        yaxis_title=f"Mean {target_col}",
        template="plotly_white",
        legend_title_text="Set",
        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,
    )

    fig.show()


## Apply the chosen splitting strategy

Here we:

- Fix the input names for splitting: `Wind Speed` and `Turbulence Intensity`.
- Apply the selected `SPLIT_METHOD` to split `df_full` into `df_train` and `df_val`.
- Keep `df_test` unchanged.

After this cell:

- `df_train`: will be used to **fit** surrogate models.
- `df_val`: will be used to **tune and compare** surrogate models (hyperparameters, model choice).
- `df_test`: remains completely unseen during training and will be used for **final evaluation**.

You can re-run this cell with different `SPLIT_METHOD` or parameters to see how the split changes.


In [ ]:
# CELL 5: Apply the chosen split and combine sets for plotting

# ============================
# Apply the chosen split
# ============================

wsp_col = "Wind Speed"
ti_col  = "Turbulence Intensity"

if SPLIT_METHOD == "random":
    df_train, df_val = split_random(
        df_full,
        train_fraction=TRAIN_FRACTION,
        random_state=RANDOM_STATE,
    )

elif SPLIT_METHOD == "holdout_wsp":
    df_train, df_val = split_holdout_range(
        df_full,
        col_name=wsp_col,
        vmin=HOLDOUT_WSP_RANGE[0],
        vmax=HOLDOUT_WSP_RANGE[1],
    )

elif SPLIT_METHOD == "holdout_ti":
    df_train, df_val = split_holdout_range(
        df_full,
        col_name=ti_col,
        vmin=HOLDOUT_TI_RANGE[0],
        vmax=HOLDOUT_TI_RANGE[1],
    )

elif SPLIT_METHOD == "stratified_target":
    df_train, df_val = split_stratified_target(
        df_full,
        target_col=TARGET_COL,
        train_fraction=TRAIN_FRACTION,
        nbins=TARGET_STRAT_NBINS,
        random_state=RANDOM_STATE,
    )

elif SPLIT_METHOD == "stratified_joint":
    df_train, df_val = split_stratified_joint(
        df_full,
        wsp_col=wsp_col,
        ti_col=ti_col,
        target_col=TARGET_COL,
        train_fraction=TRAIN_FRACTION,
        n_wsp_bins=WSP_STRAT_NBINS,
        n_ti_bins=TI_STRAT_NBINS,
        n_target_bins=TARGET_JOINT_NBINS,
        random_state=RANDOM_STATE,
    )

else:
    raise ValueError(f"Unknown SPLIT_METHOD: {SPLIT_METHOD}")

print("\nFinal split sizes:")
print("  Train:", df_train.shape)
print("  Val  :", df_val.shape)
print("  Test :", df_test.shape)

# =========================================
# Tag sets and combine for easier manipulation and plotting
# =========================================

df_train = df_train.copy()
df_val   = df_val.copy()
df_test  = df_test.copy()

df_train["set"] = "train"
df_val["set"]   = "val"
df_test["set"]  = "test"

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)



Final split sizes:
  Train: (230, 92)
  Val  : (99, 92)
  Test : (94, 92)


In [ ]:
# CELL 6: Function definitions for correlation matrices and descriptive stats


import plotly.express as px

def show_correlations(df, input_cols, target_col, set_name):
    """
    Compute and display Pearson correlation matrix between selected inputs
    and the target in a given set (train/val/test), as a heatmap with colorbar.

    Parameters
    ----------
    df : pandas.DataFrame
        Data frame containing all columns.
    input_cols : list of str
        Names of input features.
    target_col : str
        Name of target column.
    set_name : str
        Label for printing (e.g. 'train', 'val', 'test').
    """
    cols = list(input_cols) + [target_col]
    corr = df[cols].corr()

    print(f"\n=== Correlations in {set_name} set ===")

    fig = px.imshow(
        corr,
        text_auto=".2f",                # show correlation values in cells
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        width=700,
        height=400,
    )

    fig.update_layout(
        title=f"Correlation matrix ({set_name} set)",
        xaxis_title="Variables",
        yaxis_title="Variables",
        template="plotly_white",
        # coloraxis_colorbar=dict(
        #     title="Correlation",
        # ),
    )

    # Make text smaller so it fits
    fig.update_traces(
        textfont=dict(size=10)
    )

    fig.show()



def show_basic_stats(df, cols, set_name):
    """
    Print basic descriptive statistics (count, mean, std, min, quartiles, max)
    for selected columns in a given set.
    """
    print(f"\n=== Basic stats in {set_name} set ===")
    display(df[cols].describe().T)


## Correlation matrices and basic descriptive statistics

In this section we:

- Compute **Pearson correlation matrices** between the selected inputs (`SELECTED_INPUT_COLS`) and the current `TARGET_COL`.
- Print **basic descriptive statistics** (count, mean, std, min, quartiles, max) for the same variables.

What to look at:

- **Input–input correlations**:
  - High correlation between two inputs (e.g. Wind Speed and RotSpd_mean) can indicate potential redundancy (they carry similar information).
- **Input–target correlations**:
  - High correlation suggests a strong **linear** relationship with the target.
  - Low correlation does *not* mean the variable is useless; the relationship may be nonlinear.
- **Basic stats per set**:
  - Do train, val, and test cover similar ranges in Wind Speed, TI, and the target?
  - Are there shifts in means or ranges? This tells us if the model will be extrapolating on the validation or test set.


In [ ]:
# CELL 7: Show correlations

# =========================================
# Run correlations and stats for current target
# =========================================

cols_for_stats = SELECTED_INPUT_COLS + [TARGET_COL]

show_correlations(df_train, SELECTED_INPUT_COLS, TARGET_COL, set_name="train")
show_correlations(df_val,   SELECTED_INPUT_COLS, TARGET_COL, set_name="val")
show_correlations(df_test,  SELECTED_INPUT_COLS, TARGET_COL, set_name="test")





=== Correlations in train set ===



=== Correlations in val set ===



=== Correlations in test set ===


In [ ]:
# CELL 8: Show basic stats

show_basic_stats(df_train, cols_for_stats, set_name="train")
show_basic_stats(df_val,   cols_for_stats, set_name="val")
show_basic_stats(df_test,  cols_for_stats, set_name="test")


=== Basic stats in train set ===


,count,mean,std,min,25%,50%,75%,max
Wind Speed,230.0,13.269565,6.141450,3.000000,8.000000,13.000000,18.000000,26.000000
Turbulence Intensity,230.0,16.434783,8.719431,3.000000,9.000000,16.000000,25.000000,31.000000
RotSpd_mean,230.0,10.462273,2.375366,3.770073,10.255856,11.739456,11.750668,11.753320
BRFW_DEL10,230.0,5603.415784,3261.406574,249.945169,3065.979429,5287.376537,7958.499802,13325.275763



=== Basic stats in val set ===


,count,mean,std,min,25%,50%,75%,max
Wind Speed,99.0,14.121212,7.070018,3.000000,7.000000,15.000000,20.500000,26.000000
Turbulence Intensity,99.0,16.252525,8.430198,3.000000,9.000000,17.000000,23.000000,31.000000
RotSpd_mean,99.0,10.305351,2.407139,3.773138,8.983039,11.737198,11.749819,11.753363
BRFW_DEL10,99.0,5729.429453,3468.925724,288.883918,2997.699303,5335.679182,7930.867695,13765.722318



=== Basic stats in test set ===


,count,mean,std,min,25%,50%,75%,max
Wind Speed,94.0,13.840426,6.307516,3.190000,8.497500,13.825000,19.150000,25.780000
Turbulence Intensity,94.0,16.817021,8.212155,3.100000,9.525000,17.050000,23.900000,31.000000
RotSpd_mean,94.0,10.626238,2.146651,4.246244,10.751992,11.741901,11.750875,11.804141
BRFW_DEL10,94.0,5692.108719,3213.042033,285.413425,3241.664907,5610.382245,7418.602363,12746.531438


## Simple linear regression with standardized inputs

Here we fit a very simple **linear regression baseline**:

- Inputs: the selected columns in `SELECTED_INPUT_COLS` (e.g. Wind Speed, TI, RotSpd_mean).
- Target: the current `TARGET_COL` (e.g. Power or a DEL metric).
- We **standardize** the inputs (zero mean, unit variance) based on the training set.
- We fit a linear model `y ≈ β₀ + Σ βⱼ xⱼ` on the training data.
- We report:
  - R² on the training set,
  - R² on the validation set,
  - The linear coefficients for each input.

How to interpret:

- **R² values**:
  - Measure how much of the variance in the target the linear model explains.
  - If train and val R² are similar, the model generalizes reasonably for a **linear** model.
- **Coefficients**:
  - Inputs have been standardized, so each coefficient indicates the effect of a **1 standard deviation change** in that input on the predicted target (in target units).
  - Larger |coef| → stronger linear influence (given the other inputs in the model).
  - If inputs are strongly correlated (e.g. Wind Speed and RotSpd_mean), coefficients can be unstable or hard to interpret because the model is trying to disentangle overlapping effects (**multicollinearity**).

This model is *not* our final surrogate; it is a simple baseline and a way to discuss feature relevance and redundancy.


In [ ]:
# CELL 9: Fit a simple linear regression model with standardized inputs

# =========================================
# Simple linear regression with standardized inputs
# =========================================

def fit_linear_model(df_train, df_val, input_cols, target_col):
    """
    Fit a simple linear regression model on standardized inputs to predict target_col.

    - Standardizes inputs (zero mean, unit variance) based on train set.
    - Fits LinearRegression on train.
    - Reports R^2 on train and val.
    - Returns the fitted scaler and model.
    """
    X_train = df_train[input_cols].values
    y_train = df_train[target_col].values

    X_val = df_val[input_cols].values
    y_val = df_val[target_col].values

    # Standardize inputs based on train set
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    # Fit linear model
    linreg = LinearRegression()
    linreg.fit(X_train_scaled, y_train)

    # Compute R^2 on train and val
    r2_train = linreg.score(X_train_scaled, y_train)
    r2_val   = linreg.score(X_val_scaled,   y_val)

    print(f"\n=== Linear regression for target: {target_col} ===")
    print(f"Inputs: {input_cols}")
    print(f"R^2 (train): {r2_train:.3f}")
    print(f"R^2 (val)  : {r2_val:.3f}")

    # Build a small table for coefficients
    coef_table = pd.DataFrame({
        "input": input_cols,
        "coef":  linreg.coef_,
    }).sort_values(by="coef", key=np.abs, ascending=False)

    print("\nStandardized coefficients (sorted by |coef|):")
    display(coef_table)

    return scaler, linreg



In [ ]:
#CELL 10: Fit and inspect linear model

# =========================================
# Fit and inspect linear model for current target
# =========================================

scaler_lin, linreg_model = fit_linear_model(
    df_train,
    df_val,
    input_cols=SELECTED_INPUT_COLS,
    target_col=TARGET_COL,
)



=== Linear regression for target: BRFW_DEL10 ===
Inputs: ['Wind Speed', 'Turbulence Intensity', 'RotSpd_mean']
R^2 (train): 0.924
R^2 (val)  : 0.919

Standardized coefficients (sorted by |coef|):


,input,coef
1,Turbulence Intensity,2229.870299
0,Wind Speed,1778.218135
2,RotSpd_mean,755.800663


## Stratified K-fold cross-validation (demo)

This section **demonstrates** how K-fold cross-validation works for a regression problem:

- We bin the target (`TARGET_COL`) into a number of quantile bins.
- We use `StratifiedKFold` to create several (train, validation) splits.
- For each fold, we print:
  - Train and validation sizes,
  - Basic statistics of the target (mean, min, max) in each split.

This is *not* used directly in the main workflow yet, but it shows the idea of:

- Using multiple splits instead of one fixed train/val split.
- Preserving the target distribution in each fold using stratification.

Later, we could use similar ideas when tuning hyperparameters in the surrogate training notebook.


In [ ]:
# CELL 11: K-fold / stratified K-fold functions

# =========================================
# K-fold / stratified K-fold demonstration
# =========================================

def demo_stratified_kfold(df, target_col, n_splits=5, nbins=8, random_state=42):
    """
    Demonstrate stratified K-fold splitting for a regression target.

    - Bins the target into nbins quantile bins.
    - Uses StratifiedKFold to create n_splits (train, val) partitions.
    - Prints the size of each fold and optional basic stats.
    """
    y = df[target_col]
    bins = make_quantile_bins(y, nbins)  # reuse function from splitting section

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    print(f"\n=== Stratified K-fold demo for target: {target_col} ===")
    print(f"Number of splits: {n_splits}, bins: {nbins}")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, bins), start=1):
        df_tr = df.iloc[train_idx]
        df_va = df.iloc[val_idx]

        print(f"\nFold {fold_idx}:")
        print("  Train size:", len(df_tr))
        print("  Val size  :", len(df_va))

        # Quick target stats per fold (mean and range)
        print("  Train target mean/min/max:",
              df_tr[target_col].mean(),
              df_tr[target_col].min(),
              df_tr[target_col].max())
        print("  Val   target mean/min/max:",
              df_va[target_col].mean(),
              df_va[target_col].min(),
              df_va[target_col].max())


In [ ]:
# CELL 12: Run K-fold demo on the full factorial dataset for current target

# =========================================
# Run K-fold demo on the full factorial dataset for current target
# =========================================

demo_stratified_kfold(
    df_full,
    target_col=TARGET_COL,
    n_splits=5,
    nbins=8,   # reuse same nbins as for stratified split
    random_state=RANDOM_STATE,
)



=== Stratified K-fold demo for target: BRFW_DEL10 ===
Number of splits: 5, bins: 8

Fold 1:
  Train size: 263
  Val size  : 66
  Train target mean/min/max: 5662.002045839718 249.9451689445252 13765.722318235688
  Val   target mean/min/max: 5558.978912969932 337.10715671128634 12151.727611802327

Fold 2:
  Train size: 263
  Val size  : 66
  Train target mean/min/max: 5631.774428628458 249.9451689445252 13765.722318235688
  Val   target mean/min/max: 5679.4313876148035 288.883917785189 13325.275763345797

Fold 3:
  Train size: 263
  Val size  : 66
  Train target mean/min/max: 5645.392631470708 283.2888725903444 13765.722318235688
  Val   target mean/min/max: 5625.164912652502 249.9451689445252 12417.522631279217

Fold 4:
  Train size: 263
  Val size  : 66
  Train target mean/min/max: 5616.758069806249 249.9451689445252 13325.275763345797
  Val   target mean/min/max: 5739.269302315427 283.2888725903444 13765.722318235688

Fold 5:
  Train size: 264
  Val size  : 65
  Train target mean/min

## Plot 1 – Input space: Wind Speed vs Turbulence Intensity

This interactive plot shows the **input space** `(Wind Speed, Turbulence Intensity)`:

- Each point is a simulation.
- Color/marker indicates which set the sample belongs to:
  - train / val / test.

What to look for:

- How does the **full-factorial grid** (train+val) compare to the **randomized test** points?
- With different `SPLIT_METHOD` choices:
  - Does the validation set cover a specific band (for holdout splits)?
  - Are there regions where only train or only val has data?

You can zoom/pan and toggle sets on/off by clicking on the legend.


In [ ]:
# CELL 13: Plotting the input space scatter (WS vs TI) for train/val/test

# =========================================
# Plot 1: Input space scatter (WS vs TI)
# =========================================

plot_input_scatter_plotly(
    df_all,
    wsp_col="Wind Speed",
    ti_col="Turbulence Intensity",
    set_col="set",
    title=f"{SELECTED_REGIME} – Wind Speed vs TI (train/val/test)",
)


## Plot 2 – Distribution of inputs and outputs (histogram)
These plots show overlaid histograms of inputs and outputs for train, val, and test.

What to look for:

- Are the distributions similar between train, val, and test?
- Does a particular split method (e.g. holdout_wsp) create a strong bias in the validation distribution?

You can toggle each set on/off in the legend to focus on individual distributions.


In [ ]:
# CELL 14: Plotting the input histograms for WS and TI for train/val/test

# =========================================
# Plot 2: Input histograms (WS and TI)
# =========================================
plot_wsp_histogram_plotly(
    df_train,
    df_val,
    df_test,
    wsp_col="Wind Speed",
    nbins=60,
)


plot_ti_histogram_plotly(
    df_train,
    df_val,
    df_test,
    ti_col="Turbulence Intensity",
    nbins=60,
)

# =========================================
# Plot 3: Target histogram (current TARGET_COL)
# =========================================

print(f"Current target: {TARGET_COL}")

plot_target_histogram_plotly(
    df_train,
    df_val,
    df_test,
    target_col=TARGET_COL,
    nbins=60,
)


Current target: BRFW_DEL10


## Plot 3 – Target vs inputs (Wind Speed and TI)

These two scatter plots show how the current target depends on the inputs:

1. Target vs **Wind Speed**, colored by **Turbulence Intensity**.
2. Target vs **Turbulence Intensity**, colored by **Wind Speed**.

Each dataset (train/val/test) is plotted with a different marker:

- circles: train
- x: val
- triangles: test

What to look for:

- Do the points follow physically meaningful trends? (e.g. power curve shape, loads increasing with TI)
- Are there regions in Wind Speed / TI where only one set (e.g. val or test) has points?
- Are there apparent nonlinearities that a simple linear model cannot capture?

Toggling sets on/off in the legend can help isolate the distributions.


In [ ]:
# CELL 15: Plotting target vs inputs (WS and TI) for train/val/test

# =========================================
# Plot 3: Target vs inputs (WS and TI)
# =========================================


print(f"Current target: {TARGET_COL}")

plot_target_vs_inputs_plotly(
    df_all,
    target_col=TARGET_COL,
    wsp_col="Wind Speed",
    ti_col="Turbulence Intensity",
)



Current target: BRFW_DEL10


## Plot 4 – Binned average of target vs Wind Speed

Here we compute and plot the **mean target value in wind speed bins** for each set (train/val/test):

- Wind Speed is divided into equal-width bins.
- For each bin and each set, we average the target.
- We plot these binned means as lines.

What to look for:

- Do the curves for train, val, and test follow similar trends?
- Does the validation curve cover the same WS range as train?
- Are there WS regions where one set is clearly missing?

This plot is a simple way to compare “response curves” computed from different splits.


In [ ]:
# CELL 16: Plotting binned target vs Wind Speed for train/val/test


# =========================================
# Plot 4: Binned mean(target) vs Wind Speed per set
# =========================================

print(f"Current target: {TARGET_COL}")

plot_binned_target_vs_wsp_plotly(
    df_all,
    target_col=TARGET_COL,
    wsp_col="Wind Speed",
    set_col="set",
    nbins=20,
)


Current target: BRFW_DEL10
